# 📊 Visualization in Climatology Engine

This notebook introduces various visualization methods for results.

**What you will learn:**
- Loading results from Zarr files
- Plotting time series
- Plotting spatial maps of distributions
- Plotting histograms and fitted distribution curves
- Plotting comparison charts
- Using Cartopy for professional maps
- Saving figures in various formats

---

## 📐 Introduction

Data visualization is one of the most important aspects of data analysis. In this notebook we work with:

- `matplotlib`: base visualization library
- `seaborn`: advanced statistical visualization
- `cartopy`: geographic maps (optional)
- `plotly`: interactive charts (optional)

---

In [ ]:
import sys
import os
project_root = os.path.abspath('..')
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Display settings
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_context('notebook', font_scale=1.2)
%matplotlib inline

print('✅ Libraries loaded.')

In [ ]:
# Check for Cartopy
try:
    import cartopy.crs as ccrs
    import cartopy.feature as cfeature
    cartopy_available = True
    print('✅ Cartopy is available.')
except ImportError:
    cartopy_available = False
    print('⚠️ Cartopy not installed. To install: pip install cartopy')

# Check for Plotly
try:
    import plotly.express as px
    import plotly.graph_objects as go
    plotly_available = True
    print('✅ Plotly is available.')
except ImportError:
    plotly_available = False
    print('⚠️ Plotly not installed. To install: pip install plotly')

In [ ]:
# Load sample data and fit distributions (to create data for visualization)
from core.engine.plugin_loader import load_plugins

plugins = load_plugins()
distributions = {dist.name: dist for dist in plugins.values()}

sample_dir = os.path.join(project_root, 'sample_data')
station_files = sorted([f for f in os.listdir(sample_dir) if f.endswith('.csv')])
station_data = pd.read_csv(os.path.join(sample_dir, station_files[0]))
data = station_data.values
data_year = data[:365, 1]  # tmean

# Fit all distributions
results = {}
for name, dist in distributions.items():
    try:
        results[name] = dist.fit(data_year)
    except:
        pass

print(f"✅ {len(results)} distributions fitted.")

In [ ]:
# ============================================================================
# 1. Time Series Plot
# ============================================================================

fig, ax = plt.subplots(figsize=(14, 6))

ax.plot(data_year, color='blue', alpha=0.7, linewidth=1.5, label='Daily data')
ax.axhline(np.mean(data_year), color='red', linestyle='--', linewidth=2, label=f'Mean = {np.mean(data_year):.2f}°C')

ax.set_xlabel('Day of Year', fontsize=12)
ax.set_ylabel('Temperature (°C)', fontsize=12)
ax.set_title('Daily Mean Temperature Time Series', fontsize=14, fontweight='bold')
ax.legend(loc='upper right', fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================================
# 2. Histogram and Fitted Distribution Curves
# ============================================================================

fig, ax = plt.subplots(figsize=(12, 7))

# Data histogram
ax.hist(data_year, bins=30, density=True, alpha=0.4, color='gray', edgecolor='black', label='Data')

# Distribution curves
colors = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12', '#9b59b6']
x = np.linspace(min(data_year), max(data_year), 500)

for i, (name, res) in enumerate(results.items()):
    dist = distributions[name]
    if hasattr(dist, 'pdf'):
        try:
            params = {p: res[p] for p in dist.params if p in res}
            pdf_vals = dist.pdf(x, params)
            ax.plot(x, pdf_vals, color=colors[i % len(colors)], 
                    linewidth=2.5, label=f'{name} (AICc={res["aicc"]:.1f})')
        except:
            pass

ax.set_xlabel('Temperature (°C)', fontsize=12)
ax.set_ylabel('Probability Density', fontsize=12)
ax.set_title('Histogram and Fitted Distribution Curves', fontsize=14, fontweight='bold')
ax.legend(loc='upper right', fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================================
# 3. QQ-Plot (Normal Comparison)
# ============================================================================

from scipy import stats

fig, ax = plt.subplots(figsize=(8, 8))

stats.probplot(data_year, dist='norm', plot=ax)

ax.set_title('QQ-Plot (Comparison with Normal Distribution)', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================================
# 4. Spatial Map of Distributions (using synthetic data)
# ============================================================================

np.random.seed(42)
n_stations = 50
lats = np.random.uniform(25, 40, n_stations)
lons = np.random.uniform(44, 64, n_stations)
dist_codes = np.random.choice([0, 1, 2, 3, 4], n_stations)
dist_names = ['Normal', 'Skew', 'GEV', 'Bimodal', 'Pearson']

df_map = pd.DataFrame({
    'lat': lats,
    'lon': lons,
    'distribution': [dist_names[c] for c in dist_codes],
    'code': dist_codes
})

print(f"📊 {len(df_map)} points with coordinates generated.")

In [ ]:
# Simple scatter map
fig, ax = plt.subplots(figsize=(12, 10))

colors = ['#3498db', '#2ecc71', '#e74c3c', '#f39c12', '#9b59b6']

for i, dist_name in enumerate(dist_names):
    mask = df_map['distribution'] == dist_name
    if mask.any():
        ax.scatter(df_map.loc[mask, 'lon'], df_map.loc[mask, 'lat'], 
                   c=colors[i], s=100, alpha=0.7, edgecolor='black', linewidth=1,
                   label=dist_name)

ax.set_xlabel('Longitude', fontsize=12)
ax.set_ylabel('Latitude', fontsize=12)
ax.set_title('Spatial Map of Best-Fit Distributions', fontsize=14, fontweight='bold')
ax.legend(loc='upper right', fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Map with Cartopy (if available)
if cartopy_available:
    fig, ax = plt.subplots(figsize=(14, 10), subplot_kw={'projection': ccrs.PlateCarree()})
    
    ax.add_feature(cfeature.LAND, facecolor='lightgray')
    ax.add_feature(cfeature.OCEAN, facecolor='lightblue')
    ax.add_feature(cfeature.COASTLINE, linewidth=0.5)
    ax.add_feature(cfeature.BORDERS, linewidth=0.5, linestyle=':')
    ax.add_feature(cfeature.LAKES, facecolor='lightblue', alpha=0.5)
    ax.add_feature(cfeature.RIVERS, linewidth=0.3)
    
    for i, dist_name in enumerate(dist_names):
        mask = df_map['distribution'] == dist_name
        if mask.any():
            ax.scatter(df_map.loc[mask, 'lon'], df_map.loc[mask, 'lat'], 
                       transform=ccrs.PlateCarree(),
                       c=colors[i], s=120, alpha=0.8, edgecolor='black', linewidth=1,
                       label=dist_name)
    
    ax.set_extent([44, 64, 25, 40], crs=ccrs.PlateCarree())
    ax.set_title('Spatial Map of Best-Fit Distributions (with Cartopy)', fontsize=14, fontweight='bold')
    ax.legend(loc='upper right', fontsize=11)
    
    plt.tight_layout()
    plt.show()
else:
    print("⚠️ Cartopy not available. Install Cartopy for professional maps.")

In [ ]:
# ============================================================================
# 5. Interactive Maps with Plotly (Optional)
# ============================================================================

if plotly_available:
    fig = px.scatter_mapbox(
        df_map,
        lat='lat',
        lon='lon',
        color='distribution',
        hover_data={'code': True},
        color_discrete_sequence=px.colors.qualitative.Set1,
        zoom=5,
        height=600,
        title='Interactive Distribution Map'
    )
    fig.update_layout(mapbox_style='carto-positron')
    fig.show()
else:
    print("⚠️ Plotly not available. Install Plotly for interactive maps.")

In [ ]:
# ============================================================================
# 6. Boxplot for Model Comparison
# ============================================================================

model_names = []
aicc_values = []
for name, res in results.items():
    if 'aicc' in res and not np.isnan(res['aicc']):
        model_names.append(name)
        aicc_values.append(res['aicc'])

df_aicc = pd.DataFrame({'Model': model_names, 'AICc': aicc_values})

fig, ax = plt.subplots(figsize=(10, 6))

sns.boxplot(data=df_aicc, x='Model', y='AICc', ax=ax, palette='Set2')
ax.set_title('AICc Comparison Across Models', fontsize=14, fontweight='bold')
ax.set_xlabel('Model', fontsize=12)
ax.set_ylabel('AICc', fontsize=12)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================================
# 7. Saving Figures
# ============================================================================

output_dir = os.path.join(project_root, 'visualizations')
os.makedirs(output_dir, exist_ok=True)

print(f"📁 Folder {output_dir} created for saving figures.")

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(data_year, color='blue', alpha=0.7, linewidth=1.5)
ax.set_xlabel('Day of Year')
ax.set_ylabel('Temperature (°C)')
ax.set_title('Mean Temperature Time Series')
ax.grid(True, alpha=0.3)

plt.savefig(os.path.join(output_dir, 'timeseries.png'), dpi=300, bbox_inches='tight')
plt.close()

print(f"✅ Figure saved: {os.path.join(output_dir, 'timeseries.png')}")

## 📋 Summary

In this notebook you learned:

✅ Loading results from Zarr files
✅ Plotting time series
✅ Plotting histograms and fitted distribution curves
✅ Plotting QQ-Plot
✅ Plotting spatial maps with scatter and Cartopy
✅ Plotting interactive maps with Plotly
✅ Plotting boxplots
✅ Saving figures in different formats

---

**Key Takeaways:**

1. **matplotlib** is the fundamental visualization library.
2. **seaborn** is excellent for statistical plots.
3. **Cartopy** is used for professional geographic maps.
4. **Plotly** is ideal for interactive web-based charts.
5. Always save figures with high quality (dpi=300).

---

**Next Steps:**
- Notebook 09: Custom Distribution
- Notebook 10: Advanced Usage